# RepeatsDB-direct natural and DSSP/Foldseek designed modules

This is the authoritative acquisition and boundary notebook. Natural unit coordinates are copied directly from RepeatsDB and never replaced by periodic inference. Designed boundaries require independent Biotite DSSP and Foldseek 3Di/TM-align evidence. Source notebooks contain no execution outputs; missing production shards are reported explicitly.

In [ ]:
REPO = '/home/wendai/projects/hurdler/clone_repeat_protein'
ANNOTATION_INVENTORY = '/net/scratch/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/runs/run93_repeatsdb_inventory/raw/repeatsdb_annotations.parquet'
NATURAL_CATALOG = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/natural_module_catalog.parquet'
NATURAL_MAPPINGS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/natural_module_catalog_source_mappings.parquet'
NATURAL_REGIONS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/natural_module_catalog_all_region_source_mappings.parquet'
DESIGNED_INVENTORY = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_structure_inventory_expanded.parquet'
AF3_VALIDATION = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_af3_validation.parquet'
DESIGNED_CATALOG = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_module_catalog.parquet'
DESIGNED_EXCLUSIONS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_module_catalog_exclusions.csv'
DESIGNED_CANDIDATES = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_module_catalog_boundary_candidates.parquet'
DESIGNED_UNITS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_module_catalog_unit_alignment.parquet'
DESIGNED_POSITIONS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step03_module_corpus/tables/expanded-middle-repeatsdb-foldseek-v1/designed_module_catalog_position_variability.parquet'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
VERSION = 'expanded-middle-repeatsdb-foldseek-v1'
def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
def read_optional(path):
    path = Path(path)
    if not path.is_file(): return None
    return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
run_context = {'corpus_version': VERSION, 'rules_version': 'legacy-optimized-v1', 'inputs': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': [], 'status': 'passed'}

## Boundary policy

Natural: group by canonical UniProt (else full-sequence SHA256), choose the longest annotated RepeatsDB region, sort its annotated units, and select index `(unit_count-1)//2`. DSSP/Foldseek are QC only.

Designed: scan lags 6..half-chain with eight-state DSSP and Foldseek 3Di, identify the dominant contiguous recurrent scale, globally validate adjacent fragments by Foldseek/TM-align, choose the smallest passing primitive period, then select the earlier middle copy. MAFFT defines fixed positions at ≥80% conservation.

In [ ]:
paths = [ANNOTATION_INVENTORY, NATURAL_CATALOG, NATURAL_MAPPINGS, NATURAL_REGIONS, DESIGNED_INVENTORY, AF3_VALIDATION, DESIGNED_CATALOG, DESIGNED_EXCLUSIONS, DESIGNED_CANDIDATES, DESIGNED_UNITS, DESIGNED_POSITIONS]
availability = pd.DataFrame({'artifact': [Path(p).name for p in paths], 'path': paths, 'exists': [Path(p).is_file() for p in paths], 'sha256': [sha256(p) for p in paths]})
run_context['inputs'] = dict(zip(availability.artifact, availability.sha256))
if not availability.exists.all():
    run_context['status'] = 'production_pending'
    run_context['limitations'].append('One or more Digs production/finalization artifacts are not complete; no values are imputed.')
availability

In [ ]:
annotation_inventory = read_optional(ANNOTATION_INVENTORY)
natural = read_optional(NATURAL_CATALOG)
natural_mappings = read_optional(NATURAL_MAPPINGS)
natural_regions = read_optional(NATURAL_REGIONS)
designed_inventory = read_optional(DESIGNED_INVENTORY)
af3_validation = read_optional(AF3_VALIDATION)
designed = read_optional(DESIGNED_CATALOG)
exclusions = read_optional(DESIGNED_EXCLUSIONS)
candidates = read_optional(DESIGNED_CANDIDATES)
units = read_optional(DESIGNED_UNITS)
positions = read_optional(DESIGNED_POSITIONS)
run_context['row_counts'] = {name: (None if value is None else len(value)) for name, value in {'annotations':annotation_inventory,'natural_unique_units':natural,'natural_proteins':natural_mappings,'natural_annotated_regions':natural_regions,'designed_inventory':designed_inventory,'af3_validation':af3_validation,'designed_strict_units':designed,'designed_exclusions':exclusions,'period_candidates':candidates,'aligned_units':units,'position_rows':positions}.items()}
pd.DataFrame([run_context['row_counts']])

In [ ]:
if natural is not None:
    assert natural.boundary_refinement_status.eq('source_annotation_middle_unit').all()
    assert natural.selected_module_index.eq((natural.selected_module_count.astype(int)-1)//2 + 1).all()
    assert natural.unit_sequence.str.len().eq(natural.unit_length.astype(int)).all()
    natural_qc = natural.groupby(['annotation_schema','structure_source']).agg(unique_middle_units=('unit_sequence','nunique'), proteins=('protein_key','nunique'), median_length=('unit_length','median')).reset_index()
else:
    natural_qc = pd.DataFrame({'status':['production_pending']})
natural_qc

In [ ]:
if designed_inventory is not None:
    structure_flow = designed_inventory.groupby(['family','structure_inventory_status']).size().rename('proteins').reset_index()
else:
    structure_flow = pd.DataFrame({'status':['inventory_missing']})
structure_flow

In [ ]:
if designed is not None:
    assert designed.strict_dual_evidence_passed.fillna(False).all()
    assert designed.selected_module_index.eq((designed.repeat_count.astype(int)-1)//2 + 1).all()
    designed_qc = designed[['module_id','family','period','repeat_count','selected_module_index','dssp_state_agreement','dssp_transition_agreement','foldseek_3di_identity','foldseek_median_min_tm','foldseek_median_lddt','foldseek_median_coverage']].sort_values(['family','module_id'])
else:
    designed_qc = pd.DataFrame({'status':['strict_DSSP_Foldseek_shards_pending']})
designed_qc.head(20)

In [ ]:
if candidates is not None and 'module_id' in candidates and candidates.module_id.eq('designed_THR29').any():
    thr29 = candidates.loc[candidates.module_id.eq('designed_THR29') & candidates.period.isin([23,45,68,136,204]), ['period','dssp_state_agreement','dssp_transition_agreement','foldseek_3di_identity','repeat_block_recurrence_composite','dominant_recurrence_shortlist','foldseek_global_thresholds_passed']]
else:
    thr29 = pd.DataFrame({'golden_expectation':['THR29 period 68; DSSP≈0.994; Foldseek 3Di≈0.912']})
thr29

In [ ]:
run_context['filter_flow'] = ['enumerate every RepeatsDB PDB and AlphaFoldDB annotation without class caps', 'one longest annotated region per biological protein', 'select exact earlier-middle RepeatsDB unit without inferred boundary replacement', 'match author/PDB structure then AlphaFoldDB then missing-only AF3', 'Biotite DsspApp eight-state annotation', 'Foldseek structureto3didescriptor lag self-alignment', 'dominant recurrence-scale filtering to reject local structural texture', 'global adjacent-copy Foldseek/TM-align validation', 'MAFFT fixed/variable position table', 'deduplicate exact middle AA sequence within Natural and Designed separately']
run_context['limitations'].extend(['designed rows without strict dual evidence remain in the exclusion inventory', 'AF3 is QC/boundary evidence for designed proteins only and cannot modify natural RepeatsDB coordinates'])

In [ ]:
run_context